# Can we predict next-month churn?
Target: `churn_next_month`. Plan: inspect data → check for leakage → build features → compare models with account-grouped and time-based validation → conclude.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve
from sklearn.inspection import permutation_importance
df = pd.read_csv('data.csv')
df['snapshot_month'] = pd.PeriodIndex(df['snapshot_month'], freq='M')
print(df.shape, '| accounts:', df.account_ref.nunique(), '| months:', df.snapshot_month.min(), '→', df.snapshot_month.max())
print('missing values:', int(df.isna().sum().sum()), '| duplicate account-months:', int(df.duplicated(['account_ref','snapshot_month']).sum()))
print('churn rate:', round(df.churn_next_month.mean(), 4))
df.describe().T

## Leakage check
`lifetime_months` is "total months the account was active, from the CRM export of 2026-01-05" — it is measured *after* the snapshot, so it can encode the outcome. Compare it with `tenure_months` (months so far).

In [ ]:
df['lifetime_minus_tenure'] = df.lifetime_months - df.tenure_months
print(df.groupby('churn_next_month').lifetime_minus_tenure.describe())

For every churned row `lifetime_months == tenure_months` exactly (the account ended at that snapshot), while for non-churners lifetime is at least as large. So `lifetime_months` reveals the label and would not be available at prediction time. **It is excluded from the model.** Below, how misleadingly good a model looks if it is included:

In [ ]:
base_num = ['seats','monthly_fee','tenure_months','usage_hours','support_tickets']
def make_X(d):
    X = d[['plan']+base_num].copy()
    X['fee_per_seat'] = d.monthly_fee / d.seats
    return X
y, groups = df.churn_next_month, df.account_ref
num = base_num + ['fee_per_seat']
pre = ColumnTransformer([('num', StandardScaler(), num), ('plan', OneHotEncoder(), ['plan'])])
gkf = GroupKFold(n_splits=5)

def cv_scores(model, X):
    p = cross_val_predict(model, X, y, groups=groups, cv=gkf, method='predict_proba')[:,1]
    return p, roc_auc_score(y,p), average_precision_score(y,p)

Xleak = make_X(df).assign(lifetime_months=df.lifetime_months)
pre_leak = ColumnTransformer([('num', StandardScaler(), num+['lifetime_months']), ('plan', OneHotEncoder(), ['plan'])])
_, a, ap = cv_scores(make_pipeline(pre_leak, HistGradientBoostingClassifier(random_state=0)), Xleak)
print(f'WITH leaky lifetime_months (do not trust): ROC AUC={a:.3f}  PR AUC={ap:.3f}')

## Exploration (legitimate features)
Rows are monthly snapshots, so one account appears many times. All validation therefore groups by `account_ref` so the same account is never in both train and test.

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(17,3.5))
for a_, col in zip(ax[:3], ['usage_hours','tenure_months','support_tickets']):
    q = pd.qcut(df[col], 8, duplicates='drop') if df[col].nunique()>8 else df[col]
    df.groupby(q, observed=True).churn_next_month.mean().plot(marker='o', ax=a_); a_.set_title(f'churn rate by {col}'); a_.set_ylabel('churn rate'); a_.tick_params(axis='x', rotation=45)
df.groupby('snapshot_month').churn_next_month.mean().plot(ax=ax[3]); ax[3].set_title('churn rate by month')
plt.tight_layout(); plt.show()
print(df.groupby('plan').churn_next_month.agg(['mean','size']))

## Models (5-fold, grouped by account)

In [ ]:
X = make_X(df)
models = {
 'baseline (prior)': make_pipeline(pre, DummyClassifier(strategy='prior')),
 'logistic regression': make_pipeline(pre, LogisticRegression(max_iter=1000)),
 'gradient boosting': make_pipeline(pre, HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=200, random_state=0)),
}
preds, rows = {}, []
for n, m in models.items():
    p, a, ap = cv_scores(m, X); preds[n]=p; rows.append((n, a, ap))
res = pd.DataFrame(rows, columns=['model','ROC AUC','PR AUC']).set_index('model')
print('positive rate (PR AUC of a random model):', round(y.mean(),3)); res.round(3)

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(11,4))
for n,p in preds.items():
    f,t,_ = roc_curve(y,p); ax[0].plot(f,t,label=n)
    pr,rc,_ = precision_recall_curve(y,p); ax[1].plot(rc,pr,label=n)
ax[0].plot([0,1],[0,1],'k:'); ax[0].set(xlabel='FPR',ylabel='TPR',title='ROC'); ax[1].set(xlabel='recall',ylabel='precision',title='Precision-recall'); ax[0].legend(); plt.tight_layout(); plt.show()

## Time-based check
Train on earlier months, test on the last 6 (test accounts' earlier rows are excluded from training to avoid overlap).

In [ ]:
cut = df.snapshot_month.max() - 6
test = df.snapshot_month > cut
train = (df.snapshot_month <= cut) & ~df.account_ref.isin(df.account_ref[test])
m = models['gradient boosting'].fit(X[train], y[train]); p = m.predict_proba(X[test])[:,1]
print(f'train rows={train.sum()}, test rows={test.sum()}, test churn rate={y[test].mean():.3f}')
print(f'ROC AUC={roc_auc_score(y[test],p):.3f}  PR AUC={average_precision_score(y[test],p):.3f}')

## Usefulness and drivers

In [ ]:
p = preds['gradient boosting']
order = np.argsort(-p); k = int(0.1*len(p))
print(f'Top 10% highest-risk snapshots capture {y.values[order[:k]].sum()/y.sum():.1%} of churners; precision {y.values[order[:k]].mean():.1%} vs base {y.mean():.1%} (lift {y.values[order[:k]].mean()/y.mean():.1f}x)')
m = models['gradient boosting'].fit(X, y)
pi = permutation_importance(m, X, y, scoring='roc_auc', n_repeats=5, random_state=0)
pd.Series(pi.importances_mean, index=X.columns).sort_values().plot.barh(title='Permutation importance (ROC AUC drop, in-sample fit)'); plt.show()

## Conclusions
Figures below come from a run of this code (random seeds fixed); re-run the notebook to reproduce them.

* **Leakage:** `lifetime_months` equals `tenure_months` for every churned row, so it gives a perfect (ROC AUC 1.00) but useless model. It is excluded.
* **Without it, churn is only moderately predictable.** Account-grouped 5-fold CV: ROC AUC ≈ 0.67 (logistic) / 0.68 (gradient boosting) vs 0.49 for the prior baseline; PR AUC ≈ 0.28–0.30 vs 0.15 base rate.
* **Time-based holdout is a bit better in AUC but noisy:** ROC AUC ≈ 0.70, PR AUC ≈ 0.21 (test churn rate is only 9%, so PR AUC is not comparable with CV).
* **Practical value:** the top 10% highest-risk snapshots contain ~23% of churners at ~36% precision (2.3× lift). Useful for prioritising outreach, not for confident individual calls.
* **Signals:** low usage (23% churn in the lowest usage quintile vs 9% in the highest) and more support tickets (10% with 0 tickets, 25% with 2, 37% with 3) are the strongest. Plan and tenure matter little.
* **Caveat:** churn rate swings strongly by calendar period (≈4% in Q2 2024 up to ≈35% in Q4 2024), which the features cannot explain. Month-level shocks (seasonality, pricing, campaigns) would limit accuracy on future months; worth investigating before deployment.
* **Next steps:** add per-account history features (usage trend, ticket trend over previous months), and obtain a leakage-free label of account lifetime if it was meant to be used.